# Multi-Head Attention（マルチヘッド・アテンション）

このノートブックでは、Transformer の **核心部分** である
**Multi-Head Attention** を学びます。

前回の単語埋め込みで作った 7×6 の行列が、
Multi-Head Attention の中でどのように処理されるかを追います。

## 目次
1. エンコーダの中の Multi-Head Attention（図4.15）
2. **先に結論：Multi-Head Attention は何をしているのか？**
3. Multi-Head Attention の全体像（図4.16）
4. 入出力の形状が変わらない（図4.18）
5. 3つのヘッドによる並列処理（図4.19）
6. N回の繰り返し（図4.17）
7. **難しく感じるポイントと対処法**
8. ヘッド内の4ステップ（図4.21）
9. Q・K・V の生成（図4.23, 4.24, 式4-2）
10. Q・K・V の別の解釈：情報の「圧縮」（図4.25）
11. QK^T の計算と Softmax（図4.27, 4.28, 4.29, 式4-1）
12. 書籍の数値で Q₁K₁ᵀ を実際に計算する
13. **なぜ √d_k で割るのか？（図4.32, 4.33, 式4-4, 4-5）**
14. 書籍の Q₁K₁ᵀ/√2 と Softmax 結果
15. スケーリング前後の比較ヒートマップ
16. **Step 4: Softmax × V₁（図4.34, 4.35）**
17. Softmax の結果に V を掛ける（コード）
18. 3ヘッドの結合（図4.22, 4.36）
19. まとめ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# エンコーダのどの部分を学んでいるか？（図4.15）
fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(-1, 15)
ax.set_ylim(-0.5, 2.5)
ax.axis('off')

blocks = [
    ('Input\nEmbedding', '#9E9E9E', '#F5F5F5', False),
    ('Positional\nEncoding', '#9E9E9E', '#F5F5F5', False),
    ('Multi-Head\nAttention', '#E65100', '#FFF3E0', True),   # ★ 今ここ！
    ('Add &\nNorm', '#9E9E9E', '#F5F5F5', False),
    ('Feed\nForward', '#9E9E9E', '#F5F5F5', False),
    ('Add &\nNorm', '#9E9E9E', '#F5F5F5', False),
]

for i, (label, edge_color, face_color, highlight) in enumerate(blocks):
    x = i * 2.3
    lw = 3 if highlight else 1
    rect = mpatches.FancyBboxPatch((x, 0.3), 1.8, 1.5,
                                    boxstyle='round,pad=0.1',
                                    facecolor=face_color, edgecolor=edge_color, linewidth=lw)
    ax.add_patch(rect)
    ax.text(x + 0.9, 1.05, label, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(blocks) - 1:
        ax.annotate('', xy=(x + 2.1, 1.05), xytext=(x + 1.85, 1.05),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax.text(4.6 + 0.9, 2.2, '\u2190 \u4eca\u3053\u3053\uff01', fontsize=12, fontweight='bold', color='#E65100', ha='center')
ax.set_title('\u30a8\u30f3\u30b3\u30fc\u30c0\u306e\u69cb\u6210 \u2014 Multi-Head Attention \u3092\u5b66\u3076', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1. エンコーダの中の Multi-Head Attention（図4.15）

前回のノートブックで学んだ **単語埋め込み（Input Embedding）** の結果、
7トークン × 6次元 の行列が得られました。

この行列は **Positional Encoding（位置符号化）** を加算された後、
**Multi-Head Attention** に入力されます。

Multi-Head Attention は Transformer の **最も重要な処理** です。

| 用語 | 意味 |
|------|------|
| **Multi** | 複数の |
| **Head** | 処理ユニット（ヘッド） |
| **Attention** | 「どの単語に注目するか」を計算する仕組み |

つまり、**複数のヘッド（処理ユニット）で同時に「注目すべき単語」を計算する仕組み** です。

## 先に結論：Multi-Head Attention は何をしているのか？

難しそうに見えますが、**やっていること自体はシンプル** です。

### 一言でいうと

> **「各単語が、文中の他の単語をどれだけ気にすべきかを計算して、関連ある情報を取り込む」**

### 具体例

「Mount Fuji looks **beautiful** in spring.」という文で考えると：

- 「**looks**」は「**Fuji**」と「**beautiful**」に強く注目する
  - → 「Fuji が beautiful に looks」という関係を捉える
- 「**spring**」は「**in**」に注目する
  - → 「in spring」という関係を捉える

これを **全トークンが同時に** 行うのが Self-Attention です。

### 処理の流れ（超シンプル版）

```
① 入力 X に重み行列を掛けて Q, K, V を作る（ただの行列の掛け算）
② Q と K の内積で「どの単語同士が関連あるか」のスコアを計算
③ Softmax でスコアを確率に変換
④ その確率で V の情報を重み付けして集約
```

**これだけ** です。あとはこれを3つのヘッドで並列にやって、結果をくっつけるだけ。

以下のセクションでは、この流れを1ステップずつ丁寧に追っていきます。

## 2. Multi-Head Attention の全体像（図4.16）

Multi-Head Attention の中身を詳しく見てみましょう。

### 構成要素

1. 入力データ（7×6 行列）が **3つのヘッド** に分かれて入る
2. 各ヘッドで **Q（Query）・K（Key）・V（Value）** という3つの行列を生成
3. 各ヘッドが **Self-Attention の計算** を行う
4. 3つのヘッドの出力を **結合（Concat）** して元の形状に戻す

### Q・K・V とは？

| 名前 | 役割 | 日本語の例え |
|------|------|-------------|
| **Q（Query）** | 「何を知りたいか？」を表す | 質問 |
| **K（Key）** | 「何を持っているか？」を表す | 見出し・ラベル |
| **V（Value）** | 「実際の情報」を表す | 内容・中身 |

例え話：図書館で本を探すとき
- **Q**：「機械学習の本が欲しい」（自分の質問）
- **K**：各本の背表紙のラベル（「機械学習」「料理」「歴史」）
- **V**：本の中身（実際の情報）

QとKの類似度を計算して「どの本が関連あるか」を見つけ、
関連度に応じてVの情報を重み付けして取り出します。

In [ ]:
# 図4.16: Multi-Head Attention の構造を可視化

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('\u56f34.16: Multi-Head Attention \u306e\u69cb\u9020', fontsize=15, fontweight='bold')

# --- 入力 ---
input_rect = mpatches.FancyBboxPatch((5.5, 0.3), 3, 1.2,
    boxstyle='round,pad=0.1', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(input_rect)
ax.text(7, 0.9, '\u5165\u529b\uff087\u00d76\uff09', ha='center', va='center', fontsize=12, fontweight='bold')

# --- 3つのヘッド ---
head_colors = ['#FFCDD2', '#C8E6C9', '#BBDEFB']
head_labels = ['Head 1', 'Head 2', 'Head 3']
head_x = [1.5, 5.5, 9.5]

for i, (hx, color, label) in enumerate(zip(head_x, head_colors, head_labels)):
    # ヘッドの枠
    head_box = mpatches.FancyBboxPatch((hx, 2.5), 3, 5.5,
        boxstyle='round,pad=0.15', facecolor=color, edgecolor='gray', linewidth=1.5, alpha=0.3)
    ax.add_patch(head_box)
    ax.text(hx + 1.5, 7.7, label, ha='center', fontsize=11, fontweight='bold')
    
    # 入力からの矢印
    ax.annotate('', xy=(hx + 1.5, 2.5), xytext=(7, 1.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    
    # Q, K, V の生成
    for j, (qkv, qkv_color) in enumerate(zip(['Q', 'K', 'V'], ['#EF9A9A', '#A5D6A7', '#90CAF9'])):
        qkv_rect = mpatches.FancyBboxPatch((hx + j * 0.9 + 0.15, 3.0), 0.7, 0.8,
            boxstyle='round,pad=0.05', facecolor=qkv_color, edgecolor='gray')
        ax.add_patch(qkv_rect)
        ax.text(hx + j * 0.9 + 0.5, 3.4, qkv, ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Self-Attention 計算
    attn_rect = mpatches.FancyBboxPatch((hx + 0.2, 4.5), 2.6, 1.5,
        boxstyle='round,pad=0.1', facecolor='white', edgecolor='gray', linewidth=1.5)
    ax.add_patch(attn_rect)
    ax.text(hx + 1.5, 5.5, f'softmax(Q{i+1}K{i+1}\u1d40/\u221ad\u2096)', ha='center', va='center', fontsize=8)
    ax.text(hx + 1.5, 5.0, f'\u00d7 V{i+1}', ha='center', va='center', fontsize=9)
    
    # 出力 7×2
    out_rect = mpatches.FancyBboxPatch((hx + 0.5, 6.5), 2, 0.8,
        boxstyle='round,pad=0.05', facecolor='#FFF9C4', edgecolor='#F9A825')
    ax.add_patch(out_rect)
    ax.text(hx + 1.5, 6.9, '7\u00d72', ha='center', va='center', fontsize=10, fontweight='bold')

# --- Concat ---
concat_rect = mpatches.FancyBboxPatch((4, 9), 6, 1,
    boxstyle='round,pad=0.1', facecolor='#F3E5F5', edgecolor='#7B1FA2', linewidth=2)
ax.add_patch(concat_rect)
ax.text(7, 9.5, 'Concat\uff08\u7d50\u5408\uff09\u2192 7\u00d72 + 7\u00d72 + 7\u00d72 = 7\u00d76', 
        ha='center', va='center', fontsize=11, fontweight='bold')

# ヘッド出力からConcatへの矢印
for hx in head_x:
    ax.annotate('', xy=(7, 9), xytext=(hx + 1.5, 7.3),
                arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=1.5))

# --- 出力 ---
output_rect = mpatches.FancyBboxPatch((5.5, 10.5), 3, 1.2,
    boxstyle='round,pad=0.1', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(output_rect)
ax.text(7, 11.1, '\u51fa\u529b\uff087\u00d76\uff09', ha='center', va='center', fontsize=12, fontweight='bold')
ax.annotate('', xy=(7, 10.5), xytext=(7, 10),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

plt.tight_layout()
plt.show()

print("\u30dd\u30a4\u30f3\u30c8:")
print("  1. \u5165\u529b\uff087\u00d76\uff09\u304c3\u3064\u306e\u30d8\u30c3\u30c9\u306b\u5206\u914d\u3055\u308c\u308b")
print("  2. \u5404\u30d8\u30c3\u30c9\u3067 Q\u30fbK\u30fbV \u3092\u751f\u6210\u3057\u3001Self-Attention \u3092\u8a08\u7b97")
print("  3. \u5404\u30d8\u30c3\u30c9\u306e\u51fa\u529b\uff087\u00d72\uff09\u3092\u7d50\u5408\u3057\u3066 7\u00d76 \u306b\u623b\u3059")
print("  4. \u5165\u529b\u3068\u51fa\u529b\u306e\u5f62\u72b6\u304c\u540c\u3058\uff087\u00d76\uff09")

## 3. 入出力の形状が変わらない（図4.18）

Multi-Head Attention の **最大の特徴** の1つは、
**入力と出力の行列の形状がまったく同じ** であることです。

```
入力: 7×6 行列  →  Multi-Head Attention  →  出力: 7×6 行列
```

### なぜ形状が同じである必要があるのか？

- Transformer のエンコーダでは、この処理を **N回繰り返す**（原論文では N=6）
- 繰り返すには、出力を次の入力としてそのまま使える必要がある
- だから **入力と出力の形状が同じ** でなければならない

これは後の「Add & Norm」「Feed Forward」でも同様です。

In [ ]:
# 図4.18: 入出力の形状保存を可視化

np.random.seed(42)
n_tokens = 7
embed_dim = 6
words = ["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]

# 入力行列（前回のノートブックの出力）
X_input = np.round(np.random.randn(n_tokens, embed_dim) * 0.5, 3)

# 出力行列（値は変わるが形状は同じ）
X_output = np.round(np.random.randn(n_tokens, embed_dim) * 0.5, 3)

fig, axes = plt.subplots(1, 3, figsize=(16, 5),
                         gridspec_kw={'width_ratios': [2, 1, 2]})

# 入力行列
ax = axes[0]
im1 = ax.imshow(X_input, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=10)
ax.set_xticks(range(embed_dim))
ax.set_xticklabels([f'd{i+1}' for i in range(embed_dim)], fontsize=9)
ax.set_title('\u5165\u529b: 7\u00d76', fontsize=13, fontweight='bold', color='#1565C0')
for i in range(n_tokens):
    for j in range(embed_dim):
        color = 'white' if abs(X_input[i,j]) > 0.5 else 'black'
        ax.text(j, i, f'{X_input[i,j]:.2f}', ha='center', va='center', fontsize=7, color=color)

# 中央: 矢印
ax = axes[1]
ax.axis('off')
ax.text(0.5, 0.55, 'Multi-Head\nAttention', ha='center', va='center',
        fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2))
ax.annotate('', xy=(0.85, 0.45), xytext=(0.15, 0.45),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=3))
ax.text(0.5, 0.35, '\u5f62\u72b6\u306f\u540c\u3058\uff01', ha='center', fontsize=11, color='#E65100', fontweight='bold')

# 出力行列
ax = axes[2]
im2 = ax.imshow(X_output, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=10)
ax.set_xticks(range(embed_dim))
ax.set_xticklabels([f'd{i+1}' for i in range(embed_dim)], fontsize=9)
ax.set_title('\u51fa\u529b: 7\u00d76', fontsize=13, fontweight='bold', color='#2E7D32')
for i in range(n_tokens):
    for j in range(embed_dim):
        color = 'white' if abs(X_output[i,j]) > 0.5 else 'black'
        ax.text(j, i, f'{X_output[i,j]:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.suptitle('\u56f34.18: Multi-Head Attention \u306e\u5165\u51fa\u529b\u306f\u540c\u3058\u5f62\u72b6', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\u5165\u529b\u306e\u5f62\u72b6: {X_input.shape}")
print(f"\u51fa\u529b\u306e\u5f62\u72b6: {X_output.shape}")
print("\u2192 \u5024\u306f\u5909\u308f\u308b\u304c\u3001\u5f62\u72b6\uff087\u00d76\uff09\u306f\u307e\u3063\u305f\u304f\u540c\u3058")
print("\u2192 \u3060\u304b\u3089 N \u56de\u7e70\u308a\u8fd4\u3057\u3066\u3082\u554f\u984c\u306a\u3044\uff01")

## 4. 3つのヘッドによる並列処理（図4.19）

Multi-Head Attention では、入力データを **複数のヘッドに分けて並列に処理** します。

### なぜ分割するのか？

1つの巨大な Attention で計算するより、**複数の小さな Attention で異なる視点から分析** した方が効果的です。

| ヘッド | 学習する可能性のある関係 |
|--------|------------------------|
| Head 1 | 文法的な関係（主語-動詞）|
| Head 2 | 意味的な関係（形容詞-名詞）|
| Head 3 | 位置的な関係（近い単語同士）|

### 次元の分割

書籍の例では:
- 埋め込み次元: **6**
- ヘッド数: **3**
- 各ヘッドの次元 $d_k$: **6 ÷ 3 = 2**

```
入力 7×6  →  Head1: 7×2
              Head2: 7×2   →  Concat  →  7×6
              Head3: 7×2
```

In [ ]:
# 図4.19: 3つのヘッドへの分割と結合を可視化

fig, ax = plt.subplots(figsize=(15, 7))
ax.set_xlim(0, 15)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('\u56f34.19: 3\u3064\u306e\u30d8\u30c3\u30c9\u3078\u306e\u5206\u5272\u3068\u7d50\u5408', fontsize=14, fontweight='bold')

# 入力行列 7×6
input_box = mpatches.FancyBboxPatch((0.5, 2.5), 2.5, 3,
    boxstyle='round,pad=0.1', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(input_box)
ax.text(1.75, 4.5, '\u5165\u529b', ha='center', fontsize=11, fontweight='bold')
ax.text(1.75, 3.8, '7\u00d76', ha='center', fontsize=14, fontweight='bold', color='#1565C0')
ax.text(1.75, 3.1, '(\u30c8\u30fc\u30af\u30f3\u6570\u00d7\u57cb\u8fbc\u6b21\u5143)', ha='center', fontsize=8, color='gray')

# 3つのヘッド
head_colors_bg = ['#FFCDD2', '#C8E6C9', '#BBDEFB']
head_colors_edge = ['#C62828', '#2E7D32', '#1565C0']
head_y = [5.5, 3.5, 1.5]

for i, (hy, bg, edge) in enumerate(zip(head_y, head_colors_bg, head_colors_edge)):
    # 矢印: 入力 → ヘッド
    ax.annotate('', xy=(5.2, hy + 0.5), xytext=(3.0, 4.0),
                arrowprops=dict(arrowstyle='->', color=edge, lw=1.5))
    
    # ヘッドのボックス
    head_box = mpatches.FancyBboxPatch((5.2, hy), 2.5, 1,
        boxstyle='round,pad=0.1', facecolor=bg, edgecolor=edge, linewidth=2)
    ax.add_patch(head_box)
    ax.text(6.45, hy + 0.6, f'Head {i+1}', ha='center', fontsize=10, fontweight='bold')
    ax.text(6.45, hy + 0.2, 'Attention\u8a08\u7b97', ha='center', fontsize=8)
    
    # 出力 7×2
    out_box = mpatches.FancyBboxPatch((8.5, hy + 0.1), 1.5, 0.8,
        boxstyle='round,pad=0.05', facecolor='#FFF9C4', edgecolor='#F9A825', linewidth=1.5)
    ax.add_patch(out_box)
    ax.text(9.25, hy + 0.5, '7\u00d72', ha='center', va='center', fontsize=11, fontweight='bold')
    
    # 矢印: ヘッド → 出力
    ax.annotate('', xy=(8.5, hy + 0.5), xytext=(7.7, hy + 0.5),
                arrowprops=dict(arrowstyle='->', color=edge, lw=1.5))
    
    # 矢印: 出力 → Concat
    ax.annotate('', xy=(11, 4.0), xytext=(10, hy + 0.5),
                arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=1.5))

# Concat ボックス
concat_box = mpatches.FancyBboxPatch((11, 3.2), 1.5, 1.5,
    boxstyle='round,pad=0.1', facecolor='#F3E5F5', edgecolor='#7B1FA2', linewidth=2)
ax.add_patch(concat_box)
ax.text(11.75, 4.2, 'Concat', ha='center', fontsize=10, fontweight='bold')
ax.text(11.75, 3.6, '\u7d50\u5408', ha='center', fontsize=9)

# 出力 7×6
output_box = mpatches.FancyBboxPatch((13, 3.0), 1.5, 2,
    boxstyle='round,pad=0.1', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(output_box)
ax.text(13.75, 4.3, '\u51fa\u529b', ha='center', fontsize=11, fontweight='bold')
ax.text(13.75, 3.6, '7\u00d76', ha='center', fontsize=14, fontweight='bold', color='#2E7D32')

ax.annotate('', xy=(13, 4.0), xytext=(12.5, 4.0),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# 計算式
ax.text(11.75, 1.0, '7\u00d72 + 7\u00d72 + 7\u00d72 = 7\u00d76', ha='center', fontsize=11,
        fontweight='bold', color='#7B1FA2',
        bbox=dict(boxstyle='round', facecolor='#F3E5F5', edgecolor='#7B1FA2', alpha=0.5))

plt.tight_layout()
plt.show()

print("\u2605 \u6b21\u5143\u306e\u8a08\u7b97:")
print(f"  \u57cb\u3081\u8fbc\u307f\u6b21\u5143 (d_model) = 6")
print(f"  \u30d8\u30c3\u30c9\u6570 (h)       = 3")
print(f"  \u5404\u30d8\u30c3\u30c9\u306e\u6b21\u5143 (d_k)  = d_model / h = 6 / 3 = 2")
print(f"  \u7d50\u5408\u5f8c: 2 \u00d7 3 = 6  \u2192 \u5143\u306e\u6b21\u5143\u306b\u623b\u308b\uff01")

## 5. N回の繰り返し（図4.17）

Transformer のエンコーダは、Multi-Head Attention → Add & Norm → Feed Forward → Add & Norm
のブロックを **N回繰り返し** ます。

原論文「Attention Is All You Need」では **N=6** です。

### 繰り返しの効果

| 層 | 学習する内容 |
|----|-------------|
| 1層目 | 基本的な文法構造（隣接する単語の関係）|
| 2〜3層目 | より抽象的な意味関係 |
| 4〜6層目 | 文全体の文脈・深い意味理解 |

層を重ねるほど、より高度で抽象的な特徴を捉えられるようになります。
これは CNN（畳み込みニューラルネットワーク）で層を重ねるのと同じ考え方です。

In [ ]:
# 図4.17: N回の繰り返しを可視化

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(-0.5, 9)
ax.axis('off')
ax.set_title('\u56f34.17: \u30a8\u30f3\u30b3\u30fc\u30c0\u306e N \u56de\u7e70\u308a\u8fd4\u3057\uff08N=6\uff09', fontsize=14, fontweight='bold')

# 入力
ax.text(7, 0, '\u5165\u529b\uff087\u00d76\uff09', ha='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2))

# N=6 の各層
for layer in range(6):
    y = layer * 1.2 + 1.0
    alpha = 0.4 + layer * 0.12  # 層が深いほど濃く
    
    # 層のボックス
    layer_box = mpatches.FancyBboxPatch((2, y), 10, 0.8,
        boxstyle='round,pad=0.1', facecolor='#FFF3E0', edgecolor='#E65100',
        linewidth=1.5, alpha=alpha)
    ax.add_patch(layer_box)
    
    # 各ブロック
    blocks_in_layer = ['Multi-Head\nAttention', 'Add &\nNorm', 'Feed\nForward', 'Add &\nNorm']
    block_colors = ['#FFCC80', '#FFF9C4', '#A5D6A7', '#FFF9C4']
    
    for b, (block_name, bcolor) in enumerate(zip(blocks_in_layer, block_colors)):
        bx = 2.5 + b * 2.3
        block_rect = mpatches.FancyBboxPatch((bx, y + 0.05), 2, 0.7,
            boxstyle='round,pad=0.05', facecolor=bcolor, edgecolor='gray', alpha=alpha)
        ax.add_patch(block_rect)
        ax.text(bx + 1, y + 0.4, block_name, ha='center', va='center', fontsize=6, alpha=alpha + 0.2)
    
    # 層番号
    ax.text(1.5, y + 0.4, f'\u7b2c{layer+1}\u5c64', ha='center', va='center', fontsize=9, fontweight='bold')
    
    # 矢印
    if layer > 0:
        ax.annotate('', xy=(7, y), xytext=(7, y - 0.4),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1))

# 矢印: 入力→第1層
ax.annotate('', xy=(7, 1.0), xytext=(7, 0.3),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# 出力
y_out = 8.2
ax.annotate('', xy=(7, y_out), xytext=(7, y_out - 0.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.text(7, y_out + 0.3, '\u51fa\u529b\uff087\u00d76\uff09', ha='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2))

# N=6 の注釈
ax.text(13, 4, 'N=6', fontsize=20, fontweight='bold', color='#E65100',
        bbox=dict(boxstyle='round', facecolor='#FFF3E0', edgecolor='#E65100'))

plt.tight_layout()
plt.show()

print("\u30dd\u30a4\u30f3\u30c8:")
print("  1. \u5404\u5c64\u306e\u5165\u51fa\u529b\u306f\u3059\u3079\u3066 7\u00d76\uff08\u5f62\u72b6\u304c\u4fdd\u5b58\u3055\u308c\u308b\uff09")
print("  2. \u5c64\u3092\u91cd\u306d\u308b\u307b\u3069\u3001\u3088\u308a\u9ad8\u5ea6\u306a\u7279\u5fb4\u3092\u63b4\u3081\u308b")
print("  3. \u539f\u8ad6\u6587\u3067\u306f N=6 \u3060\u304c\u3001\u30e2\u30c7\u30eb\u306b\u3088\u3063\u3066\u7570\u306a\u308b")

## ここから数式に入る前に：難しく感じるポイントと対処法

ここから先はヘッド内の計算を具体的に追っていきます。
数式や行列がたくさん出てきて混乱しやすいので、**先につまずきやすいポイントを整理** しておきます。

### 「難しい」と感じる原因と、実態

| 難しく感じるポイント | 実態（そんなに難しくない理由） |
|---------------------|-------------------------------|
| Q・K・V が3つもあって混乱する | 全部「入力 X に重み行列を掛けただけ」。作り方は3つとも同じ |
| 行列のサイズが次々変わる | 行列積のルール (m×n) × (n×p) = (m×p) で自動的に決まるだけ。暗記不要 |
| ヘッドが3つある | 同じ計算を「違う重み」で3回やって、最後に横にくっつけるだけ |
| √d_k で割る意味がわからない | 値が大きくなると Softmax が極端になるから、安定させるためのおまじない |
| 式(4-1)が長くて怖い | 分解すると「内積 → 割る → Softmax → 掛ける」の4ステップだけ |

### 行列サイズの変化を追うだけでOK

計算の中身を完璧に理解しなくても、**サイズの変化** さえ追えれば全体像がわかります：

```
X (7×6)
  → Q₁, K₁, V₁ (各 7×2)     ... 7×6 × 6×2 = 7×2
  → Q₁K₁ᵀ (7×7)              ... 7×2 × 2×7 = 7×7
  → Softmax 後も (7×7)        ... サイズ変わらない
  → × V₁ で (7×2)             ... 7×7 × 7×2 = 7×2
  → 3ヘッド結合で (7×6)       ... 7×2 を3つ横に並べる
```

最初と最後が同じ **7×6** になっていることを確認してください。
では、実際のコードで確かめていきましょう。

## 6. ヘッド内の4ステップ（図4.21）

ここからは、**1つのヘッド内** で何が起きているかを詳しく見ていきます。

### Head 1 の処理は4つのステップに分かれます

| ステップ | 処理内容 | 入力 | 出力 |
|---------|---------|------|------|
| **Step 1** | Q₁, K₁, V₁ を生成する | X (7×6) | Q₁, K₁, V₁ (各 7×2) |
| **Step 2** | Q₁K₁ᵀ を計算する | Q₁ (7×2), K₁ (7×2) | 7×7 行列 |
| **Step 3** | Softmax を適用する | 7×7 行列 | 7×7 行列（確率分布）|
| **Step 4** | V₁ を掛ける | 7×7 行列, V₁ (7×2) | 7×2 行列 |

### 式(4-1): Self-Attention の計算式

$$\text{Attention}(Q_1, K_1, V_1) = \text{softmax}\left(\frac{Q_1 K_1^T}{\sqrt{d_k}}\right) V_1$$

- $Q_1 K_1^T$：Query と Key の類似度（内積）を計算
- $\sqrt{d_k}$：スケーリング（値が大きくなりすぎないように割る）
- $\text{softmax}$：確率分布に変換（合計が1になる）
- $\times V_1$：確率に基づいて Value の情報を重み付け

In [ ]:
# 図4.21: ヘッド内の4ステップを可視化

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('\u56f34.21: Head 1 \u5185\u306e4\u30b9\u30c6\u30c3\u30d7', fontsize=14, fontweight='bold')

# 入力 X
ax.text(7, 9.5, '\u5165\u529b X\uff087\u00d76\uff09', ha='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2))

steps = [
    ('Step 1: Q\u2081, K\u2081, V\u2081 \u3092\u751f\u6210', '#FFCDD2', '#C62828',
     'X \u00d7 w\u2081Q = Q\u2081 (7\u00d72)\nX \u00d7 w\u2081K = K\u2081 (7\u00d72)\nX \u00d7 w\u2081V = V\u2081 (7\u00d72)', 7.8),
    ('Step 2: Q\u2081K\u2081\u1d40 \u3092\u8a08\u7b97', '#C8E6C9', '#2E7D32',
     'Q\u2081(7\u00d72) \u00d7 K\u2081\u1d40(2\u00d77) = 7\u00d77 \u884c\u5217', 5.9),
    ('Step 3: Softmax \u3092\u9069\u7528', '#FFF9C4', '#F9A825',
     'softmax(Q\u2081K\u2081\u1d40 / \u221ad\u2096)\n\u2192 7\u00d77 \u78ba\u7387\u884c\u5217', 4.0),
    ('Step 4: V\u2081 \u3092\u639b\u3051\u308b', '#BBDEFB', '#1565C0',
     '7\u00d77 \u00d7 V\u2081(7\u00d72) = 7\u00d72 \u884c\u5217', 2.1),
]

for i, (title, bg, edge, detail, y) in enumerate(steps):
    # ステップのボックス
    step_box = mpatches.FancyBboxPatch((2, y), 10, 1.5,
        boxstyle='round,pad=0.1', facecolor=bg, edgecolor=edge, linewidth=2, alpha=0.7)
    ax.add_patch(step_box)
    ax.text(3.5, y + 1.1, title, fontsize=11, fontweight='bold', color=edge)
    ax.text(9, y + 0.75, detail, ha='center', va='center', fontsize=9, family='monospace')
    
    # ステップ間の矢印
    if i > 0:
        ax.annotate('', xy=(7, y + 1.5), xytext=(7, y + 1.8),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# 入力→Step1 の矢印
ax.annotate('', xy=(7, 9.3), xytext=(7, 9.1),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# 出力
ax.text(7, 0.5, '\u51fa\u529b\uff087\u00d72\uff09', ha='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2))
ax.annotate('', xy=(7, 0.8), xytext=(7, 2.1),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

plt.tight_layout()
plt.show()

## 7. Q・K・V の生成（図4.23, 4.24, 式4-2）

### Step 1: Q₁, K₁, V₁ の生成

入力行列 X (7×6) に **重み行列** を掛けて、Q, K, V を作ります。

### 式(4-2)

$$X \cdot w_1^Q = Q_1$$
$$X \cdot w_1^K = K_1$$
$$X \cdot w_1^V = V_1$$

### 行列のサイズ

| 行列 | サイズ | 意味 |
|------|--------|------|
| X | 7×6 | 入力（7トークン×6次元）|
| $w_1^Q$ | 6×2 | Query用の重み行列 |
| $w_1^K$ | 6×2 | Key用の重み行列 |
| $w_1^V$ | 6×2 | Value用の重み行列 |
| Q₁ | 7×2 | Query（7トークン×2次元）|
| K₁ | 7×2 | Key（7トークン×2次元）|
| V₁ | 7×2 | Value（7トークン×2次元）|

### なぜ 6×2 の重み行列なのか？

- 入力の列数が **6** なので、重み行列の行数も **6** でなければ行列積が計算できない
- 各ヘッドの次元 $d_k$ = 6 ÷ 3 = **2** なので、重み行列の列数は **2**
- 結果: 7×6 × 6×2 = **7×2**

### 重要なポイント

- $w_1^Q$, $w_1^K$, $w_1^V$ は **すべて異なる重み行列**
- これらの重みは**学習によって最適化**される
- Head 1, Head 2, Head 3 は**それぞれ別の重み行列**を持つ

In [ ]:
# 図4.23, 4.24: Q, K, V の生成を実際に計算

np.random.seed(42)

n_tokens = 7
d_model = 6    # 埋め込み次元
n_heads = 3    # ヘッド数
d_k = d_model // n_heads  # 各ヘッドの次元 = 2

words = ["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]

# 入力行列 X (7×6)  — 前回のノートブックの出力
X = np.round(np.random.randn(n_tokens, d_model) * 0.5, 3)

print("=== Step 1: Q\u2081, K\u2081, V\u2081 \u306e\u751f\u6210 ===")
print(f"\u5165\u529b X \u306e\u5f62\u72b6: {X.shape}  (7\u30c8\u30fc\u30af\u30f3 \u00d7 6\u6b21\u5143)")
print()

# Head 1 の重み行列（6×2）
W_Q1 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)  # w_1^Q
W_K1 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)  # w_1^K
W_V1 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)  # w_1^V

print(f"w\u2081Q \u306e\u5f62\u72b6: {W_Q1.shape}  (6\u00d72)")
print(f"w\u2081K \u306e\u5f62\u72b6: {W_K1.shape}  (6\u00d72)")
print(f"w\u2081V \u306e\u5f62\u72b6: {W_V1.shape}  (6\u00d72)")
print()

# Q, K, V を計算
Q1 = X @ W_Q1   # 7×6 × 6×2 = 7×2
K1 = X @ W_K1   # 7×6 × 6×2 = 7×2
V1 = X @ W_V1   # 7×6 × 6×2 = 7×2

print("--- \u884c\u5217\u7a4d\u306e\u8a08\u7b97 ---")
print(f"X(7\u00d76) \u00d7 w\u2081Q(6\u00d72) = Q\u2081({Q1.shape[0]}\u00d7{Q1.shape[1]})")
print(f"X(7\u00d76) \u00d7 w\u2081K(6\u00d72) = K\u2081({K1.shape[0]}\u00d7{K1.shape[1]})")
print(f"X(7\u00d76) \u00d7 w\u2081V(6\u00d72) = V\u2081({V1.shape[0]}\u00d7{V1.shape[1]})")
print()

# 結果を表示
for name, matrix in [('Q\u2081', Q1), ('K\u2081', K1), ('V\u2081', V1)]:
    print(f"\n{name} ({matrix.shape[0]}\u00d7{matrix.shape[1]}):")
    header = f"{'':12s}" + "".join(f"{'d'+str(i+1):>9s}" for i in range(d_k))
    print(header)
    print("-" * len(header))
    for i, word in enumerate(words):
        vals = "".join(f"{v:9.4f}" for v in matrix[i])
        print(f"{word:12s}{vals}")

In [ ]:
# 図4.23: Q, K, V 生成の行列サイズを可視化

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

matrices_info = [
    ('Q\u2081 = X \u00d7 w\u2081Q', X, W_Q1, Q1, '#EF9A9A', 'Query'),
    ('K\u2081 = X \u00d7 w\u2081K', X, W_K1, K1, '#A5D6A7', 'Key'),
    ('V\u2081 = X \u00d7 w\u2081V', X, W_V1, V1, '#90CAF9', 'Value'),
]

for ax, (title, x_mat, w_mat, result, color, label) in zip(axes, matrices_info):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    # X の箱
    x_box = mpatches.FancyBboxPatch((0.5, 1.5), 2, 3,
        boxstyle='round,pad=0.1', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=1.5)
    ax.add_patch(x_box)
    ax.text(1.5, 3.5, 'X', ha='center', fontsize=14, fontweight='bold')
    ax.text(1.5, 2.5, '7\u00d76', ha='center', fontsize=11, color='#1565C0')
    
    # ×記号
    ax.text(3.2, 3, '\u00d7', ha='center', fontsize=16, fontweight='bold')
    
    # w の箱
    w_box = mpatches.FancyBboxPatch((3.8, 2), 1.5, 2,
        boxstyle='round,pad=0.1', facecolor='#FFF9C4', edgecolor='#F9A825', linewidth=1.5)
    ax.add_patch(w_box)
    ax.text(4.55, 3.3, f'w\u2081{label[0]}', ha='center', fontsize=12, fontweight='bold')
    ax.text(4.55, 2.5, '6\u00d72', ha='center', fontsize=11, color='#F9A825')
    
    # = 記号
    ax.text(6, 3, '=', ha='center', fontsize=16, fontweight='bold')
    
    # 結果の箱
    r_box = mpatches.FancyBboxPatch((6.5, 1.5), 2, 3,
        boxstyle='round,pad=0.1', facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(r_box)
    ax.text(7.5, 3.5, f'{label[0]}\u2081', ha='center', fontsize=14, fontweight='bold')
    ax.text(7.5, 2.5, '7\u00d72', ha='center', fontsize=11)
    
    # 次元の計算
    ax.text(5, 0.8, '7\u00d7\u03366\u0336 \u00d7 \u03366\u0336\u00d72 = 7\u00d72', ha='center', fontsize=10,
            color='gray', style='italic')

plt.suptitle('\u56f34.23: Q\u30fbK\u30fbV \u306e\u751f\u6210\uff08\u884c\u5217\u306e\u30b5\u30a4\u30ba\uff09', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\u2605 \u884c\u5217\u7a4d\u306e\u30eb\u30fc\u30eb:")
print("  (m\u00d7n) \u00d7 (n\u00d7p) = (m\u00d7p)")
print("  \u5185\u5074\u306e\u6b21\u5143\uff08n\uff09\u304c\u4e00\u81f4\u3057\u306a\u3044\u3068\u639b\u3051\u7b97\u3067\u304d\u306a\u3044")
print(f"  (7\u00d76) \u00d7 (6\u00d72) = (7\u00d72)  ... \u5185\u5074\u306e 6 \u304c\u4e00\u81f4 \u2713")

### Q・K・V の別の解釈：情報の「圧縮」（図4.25）

Q₁, K₁, V₁ の生成は、見方を変えると **情報の圧縮** とも言えます。

```
入力 X (7×6)  →  重み行列 (6×2) を掛ける  →  Q₁, K₁, V₁ (各 7×2)
```

6次元 → 2次元に減っているので、**重要な要素に絞って情報を圧縮** しているのです。

| 解釈 | 説明 |
|------|------|
| **多角的な考察** | Q, K, V それぞれ異なる重み行列で変形するため、同じ入力 X を3つの異なる視点から見ている |
| **情報の圧縮** | 6次元の情報を2次元に圧縮し、重要な特徴だけを抽出している |
| **計算効率** | 次元を小さくすることで、後の計算（内積など）が軽くなる |

各ヘッドの重みは**すべて異なる値**です：

- head1 の重みパラメータ：$w_1^Q, w_1^K, w_1^V$
- head2 の重みパラメータ：$w_2^Q, w_2^K, w_2^V$
- head3 の重みパラメータ：$w_3^Q, w_3^K, w_3^V$

つまり、**Q₁, K₁, V₁ と Q₂, K₂, V₂ と Q₃, K₃, V₃ はすべて異なる値** になります。

## 8. QK^T の計算と Softmax（式4-1の前半）

### Step 2: Q₁K₁ᵀ を計算する

Q₁ と K₁ の内積（行列積）を計算します。K₁ は **転置** して掛けます。

$$Q_1 K_1^T = (7 \times 2)(2 \times 7) = 7 \times 7$$

### なぜ転置が必要なのか？（図4.27）

Q₁ と K₁ はどちらも **7×2** の行列です。
行列積のルール「(m×n) × (n×p)」を思い出してください。内側の次元が一致しないと掛け算できません。

```
Q₁(7×2) × K₁(7×2)  →  内側が 2 と 7 で不一致  →  ✕ 計算できない！

Q₁(7×2) × K₁ᵀ(2×7) →  内側が 2 と 2 で一致    →  ✓ 結果は 7×7
```

K₁ を転置して **2×7** にすることで、行列積が計算可能になります。

### Q₁K₁ᵀ の意味（図4.28, 4.29）

この 7×7 の行列は「**各トークンが他のトークンにどれだけ注目すべきか**」のスコアです。

Q₁ と K₁ᵀ はもともと入力 X を重み行列で変形したものなので、
行列積 Q₁K₁ᵀ は **各トークン同士のベクトルの内積** を計算したものだと考えられます。

| 行列の意味 | 行 | 列 |
|------------|-----|-----|
| Q₁K₁ᵀ の (i, j) 要素 | i番目のトークン（質問する側）| j番目のトークン（注目される側）|

例えば:
- 「Fuji」と「beautiful」が交差するマスは値が大きい（関連性が高い）
- 「Mount」と「in」が交差するマスは値が小さい（関連性が低い）
- 同じトークン同士のマスも関連性が高い結果になる

### Step 3: スケーリングと Softmax

内積の値が大きくなりすぎないように、$\sqrt{d_k}$ で割ります（スケーリング）。

$$\frac{Q_1 K_1^T}{\sqrt{d_k}} = \frac{Q_1 K_1^T}{\sqrt{2}} \approx \frac{Q_1 K_1^T}{1.414}$$

その後、**Softmax** を適用して確率分布に変換します。各行の合計が1になります。

In [ ]:
# Step 2 & 3: QK^T の計算 → スケーリング → Softmax

print("=== Step 2: Q\u2081K\u2081\u1d40 \u306e\u8a08\u7b97 ===")
print(f"Q\u2081 \u306e\u5f62\u72b6: {Q1.shape}  (7\u00d72)")
print(f"K\u2081\u1d40 \u306e\u5f62\u72b6: {K1.T.shape}  (2\u00d77)  \u2190 K\u2081(7\u00d72) \u3092\u8ee2\u7f6e")
print()

# Q₁K₁ᵀ を計算
QK = Q1 @ K1.T   # 7×2 × 2×7 = 7×7
print(f"Q\u2081K\u2081\u1d40 \u306e\u5f62\u72b6: {QK.shape}  (7\u00d77)")
print(f"\nQ\u2081K\u2081\u1d40 =\n{np.round(QK, 3)}")

print(f"\n=== Step 3: \u30b9\u30b1\u30fc\u30ea\u30f3\u30b0 & Softmax ===")
print(f"d_k = {d_k}")
print(f"\u221ad_k = \u221a{d_k} = {np.sqrt(d_k):.3f}")

# スケーリング
QK_scaled = QK / np.sqrt(d_k)
print(f"\nQ\u2081K\u2081\u1d40 / \u221ad_k =\n{np.round(QK_scaled, 3)}")

# Softmax
def softmax(x):
    """各行に対して Softmax を適用"""
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # オーバーフロー防止
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

attention_weights = softmax(QK_scaled)  # 7×7
print(f"\nSoftmax \u5f8c \u306e\u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u91cd\u307f:")
print(np.round(attention_weights, 3))

# 各行の合計を確認
print(f"\n\u5404\u884c\u306e\u5408\u8a08\uff08\u3059\u3079\u30661\u306b\u306a\u308b\uff09:")
for i, word in enumerate(words):
    print(f"  {word:12s}: {attention_weights[i].sum():.6f}")

In [ ]:
# 書籍(p181-182)の具体的な数値で Q₁K₁ᵀ を計算してみる

print("=== 書籍の数値例で Q₁K₁ᵀ を計算する ===")
print()

# 書籍 p181 の Q₁ と K₁ の値
Q1_book = np.array([
    [1.04, 0.19],
    [0.61, 0.69],
    [0.46, 0.45],
    [0.44, 1.10],
    [0.46, 0.82],
    [0.33, 0.65],
    [0.36, 0.92],
])

K1_book = np.array([
    [0.63, 0.27],
    [0.54, 0.44],
    [0.40, 0.40],
    [0.41, 0.59],
    [0.49, 0.24],
    [0.15, 0.65],
    [0.78, 0.56],
])

print("Q₁ (7×2):")
print(Q1_book)
print(f"\nK₁ (7×2):")
print(K1_book)

# なぜ転置が必要かを実演
print("\n--- なぜ転置が必要か？（図4.27）---")
print(f"Q₁ の形状: {Q1_book.shape}  (7×2)")
print(f"K₁ の形状: {K1_book.shape}  (7×2)")
print(f"→ Q₁(7×2) × K₁(7×2) は内側の次元が 2≠7 なので計算不可能！")
print()
print(f"K₁ᵀ の形状: {K1_book.T.shape}  (2×7)  ← 転置して行と列を入れ替え")
print(f"→ Q₁(7×2) × K₁ᵀ(2×7) は内側の次元が 2=2 で一致 → 計算可能！")

# K₁ᵀ を表示
print(f"\nK₁ᵀ (2×7):")
print(K1_book.T)

# Q₁K₁ᵀ を計算
QK_book = Q1_book @ K1_book.T   # 7×2 × 2×7 = 7×7
print(f"\nQ₁K₁ᵀ (7×7):")
print(np.round(QK_book, 2))

print("\n→ この 7×7 行列の各要素が、トークン同士の関連度スコア")
print("→ 値が大きいほど「そのトークンに注目している」ことを意味する")

In [ ]:
# 図4.29: 書籍の数値を使った Q₁K₁ᵀ のヒートマップ

fig, ax = plt.subplots(figsize=(8, 7))

# ピリオドを除いた6トークンで表示（書籍の図4.29に合わせる）
words_no_period = ["Mount", "Fuji", "looks", "beautiful", "in", "spring"]

im = ax.imshow(QK_book[:6, :6], cmap='Blues', aspect='auto')
ax.set_xticks(range(6))
ax.set_xticklabels(words_no_period, fontsize=10, rotation=45)
ax.set_yticks(range(6))
ax.set_yticklabels(words_no_period, fontsize=10)
ax.set_title('図4.29: Q₁K₁ᵀ のイメージ（トークン間の関連度）', fontsize=13, fontweight='bold')
ax.set_xlabel('注目される側 (Key)', fontsize=11)
ax.set_ylabel('質問する側 (Query)', fontsize=11)

for i in range(6):
    for j in range(6):
        val = QK_book[i, j]
        color = 'white' if val > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=ax, label='関連度スコア（色が濃い＝関連性高い）')
plt.tight_layout()
plt.show()

print("ポイント（図4.29）:")
print("  色の濃いマス → 関連性が高い（値が大きい）")
print("  色の薄いマス → 関連性が低い（値が小さい）")
print()
print("  例: 「Fuji」と「beautiful」の交差 → 値が大きい（意味的に関連）")
print("  例: 「Mount」と「in」の交差     → 値が小さい（関連性低い）")
print("  例: 同じトークン同士             → 自分自身なので関連性高い")

### なぜ $\sqrt{d_k}$ で割る必要があるのか？（図4.32, 4.33）

Q₁K₁ᵀ の値をそのまま Softmax に通すと、**値の差が極端に増幅** されてしまいます。

#### Softmax の式（式4-4）

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Softmax は指数関数 $e^x$ を使います。指数関数は **入力のわずかな差を大きく増幅** する性質があります。

#### 具体例で確認（図4.33）

| 入力 | softmax 結果 | 解釈 |
|------|-------------|------|
| [3, 2, 1] | [0.67, 0.24, 0.09] | 最大値に集中しすぎ（差が3倍→7倍に増幅）|
| [1, 2/3, 1/3] = [3,2,1]/3 | [0.45, 0.32, 0.23] | バランスが良い（差が適度に保たれる）|

$\sqrt{d_k}$ で割ることで値のスケールを小さくし、Softmax の出力が **1つの値に集中しすぎない** ようにします。

$$\frac{Q_1 K_1^T}{\sqrt{d_k}} = \frac{Q_1 K_1^T}{\sqrt{2}} \approx \frac{Q_1 K_1^T}{1.414}$$

なぜ $\sqrt{d_k}$ なのか？ → $d_k$ が大きいほど内積の値も大きくなる傾向があるため、次元数に応じたスケーリングが必要だからです。

In [ ]:
# 図4.33: スケーリングの効果を実感する

def softmax_1d(x):
    """1次元配列に対して Softmax を適用"""
    exp_x = np.exp(x - np.max(x))  # オーバーフロー防止
    return exp_x / np.sum(exp_x)

# 書籍の例: [3, 2, 1] vs [3, 2, 1] / 3
x_original = np.array([3.0, 2.0, 1.0])
x_scaled = x_original / 3.0   # √d_k で割るイメージ

sm_original = softmax_1d(x_original)
sm_scaled = softmax_1d(x_scaled)

print("=== スケーリングの効果（図4.33）===")
print()
print(f"入力 [3, 2, 1] → softmax → [{sm_original[0]:.2f}, {sm_original[1]:.2f}, {sm_original[2]:.2f}]")
print(f"  → 最大値 0.67 に集中！（1位と3位の差: {sm_original[0]/sm_original[2]:.1f}倍）")
print()
print(f"入力 [1, 2/3, 1/3] → softmax → [{sm_scaled[0]:.2f}, {sm_scaled[1]:.2f}, {sm_scaled[2]:.2f}]")
print(f"  → バランスが取れている（1位と3位の差: {sm_scaled[0]/sm_scaled[2]:.1f}倍）")
print()
print("ポイント: 割ることで softmax の出力が1つの値に集中しすぎるのを防ぐ")

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = ['x₁=3', 'x₂=2', 'x₃=1']
colors = ['#e74c3c', '#3498db', '#2ecc71']

# スケーリング前
axes[0].bar(labels, sm_original, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_title('スケーリング前: softmax([3, 2, 1])', fontsize=12, fontweight='bold')
axes[0].set_ylabel('確率', fontsize=11)
axes[0].set_ylim(0, 0.8)
for i, v in enumerate(sm_original):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=12, fontweight='bold')

# スケーリング後
labels_scaled = ['x₁=1', 'x₂=2/3', 'x₃=1/3']
axes[1].bar(labels_scaled, sm_scaled, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_title('スケーリング後: softmax([1, 2/3, 1/3])', fontsize=12, fontweight='bold')
axes[1].set_ylabel('確率', fontsize=11)
axes[1].set_ylim(0, 0.8)
for i, v in enumerate(sm_scaled):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.suptitle('図4.33: √d_k で割ることの効果', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 式(4-5): 書籍の Q₁K₁ᵀ/√2 を計算し、Softmax を適用する

print("=== 式(4-5): Q₁K₁ᵀ / √d_k の計算 ===")
print(f"d_k = {2}, √d_k = √2 = {np.sqrt(2):.4f}")
print()

# Q₁K₁ᵀ / √2
QK_book_scaled = QK_book / np.sqrt(2)
print("Q₁K₁ᵀ / √2 =")
print(np.round(QK_book_scaled, 2))

print()
print("=== Softmax を適用 ===")
print()

# Softmax（各行ごと）
def softmax_matrix(x):
    """行列の各行に Softmax を適用"""
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

attn_book = softmax_matrix(QK_book_scaled)
print("softmax(Q₁K₁ᵀ / √2) =")
print(np.round(attn_book, 2))
print()

# 各行の合計が1であることを確認
print("各行の合計（すべて1になる）:")
for i in range(7):
    print(f"  行{i} ({words[i]:12s}): {attn_book[i].sum():.6f}")

print()
print("ポイント:")
print("  Q₁K₁ᵀ → √d_k で割る → Softmax → 確率分布")
print("  各行は「そのトークンが他のトークンにどれだけ注目するか」の確率を表す")

In [ ]:
# 図4.32: スケーリング前後の比較ヒートマップ

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
words_no_period = ["Mount", "Fuji", "looks", "beautiful", "in", "spring"]

# 1. スケーリング前: Q₁K₁ᵀ
ax = axes[0]
im1 = ax.imshow(QK_book[:6, :6], cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(6))
ax.set_xticklabels(words_no_period, fontsize=9, rotation=45)
ax.set_yticks(range(6))
ax.set_yticklabels(words_no_period, fontsize=9)
ax.set_title('Q₁K₁ᵀ\n（スケーリング前）', fontsize=12, fontweight='bold')
for i in range(6):
    for j in range(6):
        ax.text(j, i, f'{QK_book[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im1, ax=ax, shrink=0.8)

# 2. スケーリング後: Q₁K₁ᵀ / √2
ax = axes[1]
im2 = ax.imshow(QK_book_scaled[:6, :6], cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(6))
ax.set_xticklabels(words_no_period, fontsize=9, rotation=45)
ax.set_yticks(range(6))
ax.set_yticklabels(words_no_period, fontsize=9)
ax.set_title('Q₁K₁ᵀ / √2\n（スケーリング後）', fontsize=12, fontweight='bold')
for i in range(6):
    for j in range(6):
        ax.text(j, i, f'{QK_book_scaled[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im2, ax=ax, shrink=0.8)

# 3. Softmax後
ax = axes[2]
im3 = ax.imshow(attn_book[:6, :6], cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.3)
ax.set_xticks(range(6))
ax.set_xticklabels(words_no_period, fontsize=9, rotation=45)
ax.set_yticks(range(6))
ax.set_yticklabels(words_no_period, fontsize=9)
ax.set_title('softmax(Q₁K₁ᵀ / √2)\n（確率分布）', fontsize=12, fontweight='bold')
for i in range(6):
    for j in range(6):
        ax.text(j, i, f'{attn_book[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im3, ax=ax, shrink=0.8)

plt.suptitle('図4.32: Q₁K₁ᵀ → スケーリング → Softmax の変換過程', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("変換の流れ:")
print("  Q₁K₁ᵀ（生の内積スコア）→ √d_k で割る（値を抑える）→ Softmax（確率分布にする）")
print("  各行の合計が 1.0 になっている → 「注目の配分」として解釈できる")

### Step 4: Softmax の結果に V₁ を掛ける（図4.34, 4.35）

ここまでで $\text{softmax}(Q_1 K_1^T / \sqrt{d_k})$ が計算できました。
これは 7×7 の「**アテンション重み行列**」です。

最後にこれを V₁（7×2）に掛けます：

$$\underbrace{\text{softmax}\left(\frac{Q_1 K_1^T}{\sqrt{d_k}}\right)}_{7 \times 7} \times \underbrace{V_1}_{7 \times 2} = \underbrace{\text{Head 1 の出力}}_{7 \times 2}$$

#### この計算の意味

アテンション重み行列の各行は「各トークンへの注目度の配分」です。
V₁ は「実際に持っている情報」です。

**注目度で重み付けして情報を集約** することで、文脈を考慮した表現が得られます。

> 書籍の言葉を借りれば「**極めて優れた数理モデルの設計**」です。

In [ ]:
# アテンション重みのヒートマップ

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左: スケーリング前の Q₁K₁ᵀ
ax = axes[0]
im1 = ax.imshow(QK, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(words, fontsize=9, rotation=45)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
ax.set_title('Q\u2081K\u2081\u1d40\uff08\u30b9\u30b1\u30fc\u30ea\u30f3\u30b0\u524d\uff09', fontsize=12, fontweight='bold')
ax.set_xlabel('\u6ce8\u76ee\u3055\u308c\u308b\u5074 (Key)', fontsize=10)
ax.set_ylabel('\u8cea\u554f\u3059\u308b\u5074 (Query)', fontsize=10)
for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f'{QK[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im1, ax=ax)

# 右: Softmax 後のアテンション重み
ax = axes[1]
im2 = ax.imshow(attention_weights, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.3)
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(words, fontsize=9, rotation=45)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
ax.set_title('Softmax\u5f8c\uff08\u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u91cd\u307f\uff09', fontsize=12, fontweight='bold')
ax.set_xlabel('\u6ce8\u76ee\u3055\u308c\u308b\u5074 (Key)', fontsize=10)
ax.set_ylabel('\u8cea\u554f\u3059\u308b\u5074 (Query)', fontsize=10)
for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f'{attention_weights[i,j]:.3f}', ha='center', va='center', fontsize=7)
plt.colorbar(im2, ax=ax)

plt.suptitle('Step 2\u30fb3: Q\u2081K\u2081\u1d40 \u2192 Softmax\uff08\u5404\u884c\u306e\u5408\u8a08=1\uff09', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\u30dd\u30a4\u30f3\u30c8:")
print("  1. Q\u2081K\u2081\u1d40 \u3067\u5404\u30c8\u30fc\u30af\u30f3\u9593\u306e\u300c\u6ce8\u76ee\u5ea6\u30b9\u30b3\u30a2\u300d\u3092\u8a08\u7b97")
print("  2. \u221ad_k \u3067\u5272\u3063\u3066\u5024\u3092\u5b89\u5b9a\u3055\u305b\u308b\uff08\u30b9\u30b1\u30fc\u30ea\u30f3\u30b0\uff09")
print("  3. Softmax \u3067\u78ba\u7387\u5206\u5e03\u306b\u5909\u63db\uff08\u5404\u884c\u306e\u5408\u8a08=1\uff09")
print("  4. \u5024\u304c\u5927\u304d\u3044\u307b\u3069\u300c\u305d\u306e\u30c8\u30fc\u30af\u30f3\u306b\u6ce8\u76ee\u3057\u3066\u3044\u308b\u300d\u3053\u3068\u3092\u610f\u5473\u3059\u308b")

## 9. Softmax の結果に V を掛ける（Step 4）

### 式(4-1) の完成

$$\text{Attention}(Q_1, K_1, V_1) = \underbrace{\text{softmax}\left(\frac{Q_1 K_1^T}{\sqrt{d_k}}\right)}_{7 \times 7} \underbrace{V_1}_{7 \times 2} = \underbrace{\text{出力}}_{7 \times 2}$$

### この計算の意味

Softmax の結果（7×7 のアテンション重み）は「各トークンがどのトークンに注目するか」の確率です。

これに V₁ を掛けることで、**注目すべきトークンの情報を重み付けして集約** します。

例えば「looks」の行の重みが [0.1, 0.3, 0.1, 0.2, 0.1, 0.1, 0.1] だった場合、
「Fuji」（2番目）の情報を最も多く取り込みます。

In [ ]:
# Step 4: Softmax結果 × V₁

print("=== Step 4: V\u2081 \u3092\u639b\u3051\u308b ===")
print(f"\u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u91cd\u307f\u306e\u5f62\u72b6: {attention_weights.shape}  (7\u00d77)")
print(f"V\u2081 \u306e\u5f62\u72b6:           {V1.shape}  (7\u00d72)")

# 最終出力
head1_output = attention_weights @ V1   # 7×7 × 7×2 = 7×2
print(f"\u51fa\u529b\u306e\u5f62\u72b6:           {head1_output.shape}  (7\u00d72)")
print()

print("--- Head 1 \u306e\u51fa\u529b ---")
header = f"{'':12s}" + "".join(f"{'d'+str(i+1):>9s}" for i in range(d_k))
print(header)
print("-" * len(header))
for i, word in enumerate(words):
    vals = "".join(f"{v:9.4f}" for v in head1_output[i])
    print(f"{word:12s}{vals}")

print(f"\n\u2605 Head 1 \u306e\u51fa\u529b\u306f 7\u00d72 \u884c\u5217")
print("\u2605 \u540c\u69d8\u306b Head 2, Head 3 \u3082\u305d\u308c\u305e\u308c 7\u00d72 \u3092\u51fa\u529b")
print("\u2605 3\u3064\u3092\u7d50\u5408\u3059\u308b\u3068 7\u00d76 \u306b\u306a\u308b\uff01")

In [ ]:
# 1つのトークンについて Step 4 の計算過程を詳しく見る

token_idx = 2  # "looks" の例
print(f'=== "{words[token_idx]}" \u306e\u8a08\u7b97\u904e\u7a0b \u3092\u8a73\u3057\u304f\u898b\u308b ===')
print()

print(f"\u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u91cd\u307f\uff08{words[token_idx]} \u306e\u884c\uff09:")
for j, word in enumerate(words):
    print(f"  {word:12s}: {attention_weights[token_idx, j]:.4f}")

print(f"\nV\u2081 \u306e\u5404\u884c:")
for j, word in enumerate(words):
    print(f"  {word:12s}: [{V1[j, 0]:7.4f}, {V1[j, 1]:7.4f}]")

print(f"\n\u8a08\u7b97: \u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u91cd\u307f \u00d7 V\u2081 \u306e\u5404\u884c \u3092\u8db3\u3057\u5408\u308f\u305b\u308b")
result_d1 = 0
result_d2 = 0
for j, word in enumerate(words):
    w = attention_weights[token_idx, j]
    contrib_d1 = w * V1[j, 0]
    contrib_d2 = w * V1[j, 1]
    result_d1 += contrib_d1
    result_d2 += contrib_d2
    print(f"  {w:.4f} \u00d7 [{V1[j, 0]:7.4f}, {V1[j, 1]:7.4f}]  =  [{contrib_d1:8.5f}, {contrib_d2:8.5f}]  ({word})")

print(f"\n\u5408\u8a08: [{result_d1:.4f}, {result_d2:.4f}]")
print(f"\u691c\u8a3c: [{head1_output[token_idx, 0]:.4f}, {head1_output[token_idx, 1]:.4f}]  \u2190 \u4e00\u81f4\uff01")
print()
print("\u2192 \u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u91cd\u307f\u304c\u5927\u304d\u3044\u30c8\u30fc\u30af\u30f3\u306e V \u304c\u3088\u308a\u5f37\u304f\u53cd\u6620\u3055\u308c\u308b")

## 13. 3ヘッドの結合（図4.22, 4.36）

Head 1, Head 2, Head 3 それぞれが同じ処理を **異なる重み行列** で行います。

各ヘッドの出力（7×2）を横に結合（Concat）して、元の 7×6 に戻します。

$$\text{MultiHead} = \text{Concat}(\text{head}_1, \text{head}_2, \text{head}_3)$$

ここで:
$$\text{head}_i = \text{softmax}\left(\frac{Q_i K_i^T}{\sqrt{d_k}}\right) V_i$$

### 結合のイメージ（図4.36）

```
Head 1 出力 (7×2)  |  Head 2 出力 (7×2)  |  Head 3 出力 (7×2)
                    ↓ 横に連結（Concat）
                MultiHead 出力 (7×6)
```

各ヘッドが **異なる観点** から文脈を捉えた結果を1つにまとめることで、
多角的な情報を持つ表現を得られます。

In [ ]:
# 図4.22: 3つのヘッドの計算と結合

# Head 2, Head 3 も計算する
np.random.seed(100)

# Head 2 の重み行列
W_Q2 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
W_K2 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
W_V2 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)

Q2 = X @ W_Q2
K2 = X @ W_K2
V2 = X @ W_V2

attn2 = softmax(Q2 @ K2.T / np.sqrt(d_k))
head2_output = attn2 @ V2

# Head 3 の重み行列
np.random.seed(200)
W_Q3 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
W_K3 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
W_V3 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)

Q3 = X @ W_Q3
K3 = X @ W_K3
V3 = X @ W_V3

attn3 = softmax(Q3 @ K3.T / np.sqrt(d_k))
head3_output = attn3 @ V3

print("=== 3\u3064\u306e\u30d8\u30c3\u30c9\u306e\u51fa\u529b ===")
print(f"Head 1 \u306e\u51fa\u529b: {head1_output.shape}")
print(f"Head 2 \u306e\u51fa\u529b: {head2_output.shape}")
print(f"Head 3 \u306e\u51fa\u529b: {head3_output.shape}")

# 結合（Concat）
multi_head_output = np.concatenate([head1_output, head2_output, head3_output], axis=1)
print(f"\n=== Concat \u5f8c ===")
print(f"\u7d50\u5408\u5f8c\u306e\u5f62\u72b6: {multi_head_output.shape}  \u2190 7\u00d72 + 7\u00d72 + 7\u00d72 = 7\u00d76")

print(f"\n--- Multi-Head Attention \u306e\u6700\u7d42\u51fa\u529b ---")
header = f"{'':12s}" + "".join(f"{'d'+str(i+1):>9s}" for i in range(d_model))
print(header)
print("-" * len(header))
for i, word in enumerate(words):
    vals = "".join(f"{v:9.4f}" for v in multi_head_output[i])
    print(f"{word:12s}{vals}")

print(f"\n\u2605 \u5165\u529b: {X.shape} \u2192 \u51fa\u529b: {multi_head_output.shape}  \u5f62\u72b6\u304c\u540c\u3058\uff01")

In [ ]:
# 3つのヘッドの結合を可視化

fig, axes = plt.subplots(1, 5, figsize=(18, 5),
                         gridspec_kw={'width_ratios': [1, 1, 1, 0.3, 2]})

head_outputs = [head1_output, head2_output, head3_output]
head_colors_cm = ['Reds', 'Greens', 'Blues']
head_names = ['Head 1\n(7\u00d72)', 'Head 2\n(7\u00d72)', 'Head 3\n(7\u00d72)']

for idx, (ax, output, cmap, name) in enumerate(zip(axes[:3], head_outputs, head_colors_cm, head_names)):
    im = ax.imshow(output, cmap=cmap, aspect='auto')
    ax.set_yticks(range(n_tokens))
    ax.set_yticklabels(words if idx == 0 else [], fontsize=9)
    ax.set_xticks(range(d_k))
    ax.set_xticklabels([f'd{i+1}' for i in range(d_k)], fontsize=8)
    ax.set_title(name, fontsize=11, fontweight='bold')
    for i in range(n_tokens):
        for j in range(d_k):
            ax.text(j, i, f'{output[i,j]:.2f}', ha='center', va='center', fontsize=7)

# 矢印
ax = axes[3]
ax.axis('off')
ax.text(0.5, 0.5, '\u2192\nConcat', ha='center', va='center', fontsize=12, fontweight='bold')

# 結合後
ax = axes[4]
im = ax.imshow(multi_head_output, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
ax.set_xticks(range(d_model))
ax.set_xticklabels([f'd{i+1}' for i in range(d_model)], fontsize=8)
ax.set_title('\u7d50\u5408\u5f8c (7\u00d76)', fontsize=11, fontweight='bold')
for i in range(n_tokens):
    for j in range(d_model):
        color = 'white' if abs(multi_head_output[i,j]) > 0.3 else 'black'
        ax.text(j, i, f'{multi_head_output[i,j]:.2f}', ha='center', va='center', fontsize=6, color=color)

# ヘッド境界線
for boundary in [2, 4]:
    ax.axvline(x=boundary - 0.5, color='white', linewidth=2)

plt.suptitle('\u56f34.22: 3\u30d8\u30c3\u30c9\u306e\u51fa\u529b\u3092\u7d50\u5408\uff08Concat\uff09', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3つのヘッドのアテンションパターンを比較

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

attentions = [attention_weights, attn2, attn3]
head_labels = ['Head 1', 'Head 2', 'Head 3']

for ax, attn, label in zip(axes, attentions, head_labels):
    im = ax.imshow(attn, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.35)
    ax.set_xticks(range(n_tokens))
    ax.set_xticklabels(words, fontsize=9, rotation=45)
    ax.set_yticks(range(n_tokens))
    ax.set_yticklabels(words, fontsize=9)
    ax.set_title(f'{label} \u306e\u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u30d1\u30bf\u30fc\u30f3', fontsize=12, fontweight='bold')
    ax.set_xlabel('Key', fontsize=10)
    ax.set_ylabel('Query', fontsize=10)
    for i in range(n_tokens):
        for j in range(n_tokens):
            ax.text(j, i, f'{attn[i,j]:.2f}', ha='center', va='center', fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('\u5404\u30d8\u30c3\u30c9\u304c\u7570\u306a\u308b\u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u30d1\u30bf\u30fc\u30f3\u3092\u5b66\u7fd2\u3059\u308b', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\u2605 \u5404\u30d8\u30c3\u30c9\u306e\u91cd\u307f\u884c\u5217\u304c\u7570\u306a\u308b\u305f\u3081\u3001\u7570\u306a\u308b\u30a2\u30c6\u30f3\u30b7\u30e7\u30f3\u30d1\u30bf\u30fc\u30f3\u306b\u306a\u308b")
print("\u2605 \u3042\u308b\u30d8\u30c3\u30c9\u306f\u6587\u6cd5\u7684\u95a2\u4fc2\u3092\u3001\u5225\u306e\u30d8\u30c3\u30c9\u306f\u610f\u5473\u7684\u95a2\u4fc2\u3092\u6355\u3048\u308b")
print("\u2605 \u8907\u6570\u306e\u8996\u70b9\u3092\u7d71\u5408\u3059\u308b\u3053\u3068\u3067\u3001\u3088\u308a\u8c4a\u304b\u306a\u8868\u73fe\u304c\u5f97\u3089\u308c\u308b")

## 19. まとめ

| ポイント | 内容 |
|----------|------|
| **Multi-Head Attention** | Transformer の核心部分。複数のヘッドで並列に Attention を計算する |
| **Q（Query）** | 「何を知りたいか？」を表す行列 |
| **K（Key）** | 「何を持っているか？」を表す行列 |
| **V（Value）** | 「実際の情報」を表す行列 |
| **情報の圧縮** | Q/K/V は入力 X を重み行列で変形した「圧縮された表現」とも解釈できる |
| **式(4-2)** | $Xw_1^Q = Q_1$, $Xw_1^K = K_1$, $Xw_1^V = V_1$ |
| **式(4-1)** | $\text{softmax}(Q_1 K_1^T / \sqrt{d_k}) V_1$ |
| **転置の必要性** | Q₁(7×2) × K₁(7×2) は計算不可。K₁ᵀ(2×7) にすれば可能 |
| **Q₁K₁ᵀ の意味** | 各トークン同士のベクトルの内積 → 関連度スコア |
| **√d_k スケーリング** | Softmax の入力値を抑え、1つの値に集中しすぎるのを防ぐ |
| **式(4-4)** | $\text{softmax}(x_i) = e^{x_i} / \sum_j e^{x_j}$ — 指数関数なので差が増幅される |
| **Step 4 の意味** | アテンション重み(7×7) × V₁(7×2) = 注目度で重み付けした情報の集約(7×2) |
| **ヘッド数** | 書籍では3（原論文では8）|
| **各ヘッドの次元 $d_k$** | $d_{model} / h$ = 6 / 3 = 2 |
| **入出力の形状** | 常に 7×6 で変わらない（N回繰り返し可能）|
| **N回繰り返し** | 原論文では N=6。層を重ねるほど高度な特徴を捉える |

### Self-Attention の全体の流れ（Head 1 の場合）

```
入力 X (7×6)
  ↓ Step 1: 重み行列を掛けて Q₁, K₁, V₁ を生成（情報を圧縮）
Q₁ (7×2), K₁ (7×2), V₁ (7×2)
  ↓ Step 2: Q₁K₁ᵀ で注目度スコアを計算（内積 = 関連度）
7×7 行列（各トークン間のスコア）
  ↓ Step 3: √d_k で割って Softmax → 確率分布（値の集中を防ぐ）
7×7 行列（各行の合計=1）
  ↓ Step 4: V₁ を掛けて情報を重み付け集約（極めて優れた数理モデルの設計）
出力 (7×2)
```

### 3ヘッドの結合（図4.36）

```
Head 1 (7×2) + Head 2 (7×2) + Head 3 (7×2)  →  Concat  →  7×6
```

## 次のステップ

次のノートブックでは **Add & Norm**（残差接続と層正規化）を学びます。

Multi-Head Attention の出力に対して：
1. **線形変換**（W_o を掛ける）
2. **Add**（元の入力 X を足す = Skip Connection）
3. **Norm**（Layer Normalization で正規化）

を行い、次の層への入力を整えます。